# Convert .pth -> TFLite (FP32 + INT8 dynamic) va danh gia lai tren tap test

Nhan truc tiep cac file `.pth` (teacher + student), convert sang TFLite **FP32** va **INT8 dynamic range**,
danh gia lai tren tap test, in ket qua theo chieu doc va luu day du ra file.

**Kien trúc duoc nhan dien theo TEN FILE** (ten cung, theo danh sach da co).

**Cach convert:** doc truc tiep `state_dict` -> dung lai model Keras tuong duong -> TFLite.
Khong phu thuoc ten class PyTorch. Da kiem chung 3 chieu doc lap:
PyTorch == numpy reference == TFLite FP32 (sai so ~1e-7).

**Voi model adaptive temperature:** lop `proj` (dau chieu) **bi loai bo** khoi ca model TFLite lan
phep do kich thuoc, vi proj chi dung luc train de tinh adaptive tau, khong nam trong duong inference.


## Cell 1 - Cau hinh

In [1]:
!pip install -q pyarrow pandas scikit-learn torch tensorflow matplotlib

import os, gc, io, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
warnings.filterwarnings('ignore')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')

# ── Duong dan ────────────────────────────────────────────────────────────────
INPUT_DIR = '/kaggle/input/datasets/rsnngo/37ft-thang3737'
TEST_FILE = os.path.join(INPUT_DIR, 'test_standard_p3.2.parquet')

PTH_DIR   = '/kaggle/input/datasets/rsnngo/pth-student-thang3737'      # <-- SUA: thu muc chua cac file .pth

OUT_DIR   = '/kaggle/working/tflite_out'
DIR_MODEL = os.path.join(OUT_DIR, 'models')
DIR_REPORT= os.path.join(OUT_DIR, 'reports')
DIR_CM    = os.path.join(OUT_DIR, 'confusion')
DIR_CHART = os.path.join(OUT_DIR, 'charts')
for d in [OUT_DIR, DIR_MODEL, DIR_REPORT, DIR_CM, DIR_CHART]:
    os.makedirs(d, exist_ok=True)

NUM_FEATURES = 37
NUM_CLASSES  = 8
LABEL_COL    = 'label'
BATCH_SIZE   = 512
LABEL_NAMES  = {0:'Benign',1:'DDoS',2:'DoS',3:'BruteForce',
                4:'Spoofing',5:'Recon',6:'Web-Based',7:'Mirai'}
CLASS_NAMES  = [LABEL_NAMES[i] for i in range(NUM_CLASSES)]
COLORS = ['#4C9BE8','#E84C4C','#F5A623','#9B59B6','#2ECC71','#1ABC9C','#8B4513','#607D8B']

# ── Danh sach file .pth (TEN CUNG) ───────────────────────────────────────────
# Moi dong: (ten file, nhan hien thi, loai)
PTH_LIST = [
    # ---- Teacher ----
    ('teacher_ssl_cl_dnn_37ft.pth',                  'Teacher SSL-CL-DNN',            'Teacher'),
    ('teacher_dnn_37ft.pth',                         'Teacher DNN',                   'Teacher'),
    ('teacher_cnn_37ft.pth',                         'Teacher CNN',                   'Teacher'),
    # ---- Student SSL-CL-DNN ----
    ('student_ssl_cl_dnn_fix_t_classw3_37ft.pth',    'SSL-CL-DNN RAID fixed tau',     'Student'),
    ('student_ssl_cl_dnn_adapt_t_classw3_37ft.pth',  'SSL-CL-DNN RAID adaptive tau',  'Student'),
    ('student_ssl_cl_dnn_vanilla_kd_classw3_37ft.pth','SSL-CL-DNN VanillaKD',         'Student'),
    ('student_ssl_cl_dnn_fitnets_37ft.pth',          'SSL-CL-DNN FitNets',            'Student'),
    # ---- Student DNN ----
    ('student_dnn_fix_t_classw3_37ft.pth',           'DNN RAID fixed tau',            'Student'),
    ('student_dnn_adapt_t_classw3_37ft.pth',         'DNN RAID adaptive tau',         'Student'),
    ('student_dnn_vanilla_kd_classw3_37ft.pth',      'DNN VanillaKD',                 'Student'),
    ('student_dnn_fitnets__classw3_37ft.pth',        'DNN FitNets',                   'Student'),
    # ---- Student CNN ----
    ('student_cnn_fix_t_classw3_37ft.pth',           'CNN RAID fixed tau',            'Student'),
    ('student_cnn_adapt_t_classw3_37ft.pth',         'CNN RAID adaptive tau',         'Student'),
    ('student_cnn_vanilla_kd_classw3_37ft.pth',      'CNN VanillaKD',                 'Student'),
    ('student_cnn_fitnets_classw3_37ft.pth',         'CNN FitNets',                   'Student'),
]

print('Thu muc .pth :', PTH_DIR)
print('Thu muc xuat :', OUT_DIR)
print()
print('Kiem tra file:')
n_found = 0
for fn, label, kind in PTH_LIST:
    p = os.path.join(PTH_DIR, fn)
    exists = os.path.exists(p)
    n_found += exists
    size = f'{os.path.getsize(p)/1024:8.1f} KB' if exists else '   --    '
    print(f'  [{"CO   " if exists else "THIEU"}] {size}  {fn}')
print()
print(f'Tim thay {n_found}/{len(PTH_LIST)} file.')
if n_found < len(PTH_LIST):
    print('Luu y: file thieu se duoc bo qua. Kiem tra lai PTH_DIR hoac ten file.')


Thu muc .pth : /kaggle/input/datasets/rsnngo/pth-student-thang3737
Thu muc xuat : /kaggle/working/tflite_out

Kiem tra file:
  [CO   ]   2530.0 KB  teacher_ssl_cl_dnn_37ft.pth
  [CO   ]    579.8 KB  teacher_dnn_37ft.pth
  [CO   ]    627.7 KB  teacher_cnn_37ft.pth
  [CO   ]     81.6 KB  student_ssl_cl_dnn_fix_t_classw3_37ft.pth
  [CO   ]    146.0 KB  student_ssl_cl_dnn_adapt_t_classw3_37ft.pth
  [CO   ]     81.7 KB  student_ssl_cl_dnn_vanilla_kd_classw3_37ft.pth
  [CO   ]     81.5 KB  student_ssl_cl_dnn_fitnets_37ft.pth
  [CO   ]     55.7 KB  student_dnn_fix_t_classw3_37ft.pth
  [CO   ]    120.0 KB  student_dnn_adapt_t_classw3_37ft.pth
  [CO   ]     55.8 KB  student_dnn_vanilla_kd_classw3_37ft.pth
  [CO   ]     55.8 KB  student_dnn_fitnets__classw3_37ft.pth
  [CO   ]     55.9 KB  student_cnn_fix_t_classw3_37ft.pth
  [CO   ]   1080.3 KB  student_cnn_adapt_t_classw3_37ft.pth
  [CO   ]     56.0 KB  student_cnn_vanilla_kd_classw3_37ft.pth
  [CO   ]     56.0 KB  student_cnn_fitnets_classw3_3

## Cell 2 - Doc tap test

In [2]:
df_test = pd.read_parquet(TEST_FILE)
feat_cols = [c for c in df_test.columns if c != LABEL_COL]
assert len(feat_cols) == NUM_FEATURES, f'{len(feat_cols)} != {NUM_FEATURES}'

X_flat = df_test[feat_cols].to_numpy(dtype=np.float32)          # (N,37)  cho DNN / SSL
X_cnn  = X_flat.reshape(-1, NUM_FEATURES, 1)                     # (N,37,1) channels-last cho Keras/TFLite
y_true = df_test[LABEL_COL].to_numpy(dtype=np.int64)

print(f'Tap test    : {len(y_true):,} mau')
print(f'X_flat shape: {X_flat.shape}')
print(f'X_cnn  shape: {X_cnn.shape}')
print()
print('Phan bo nhan:')
cnt = np.bincount(y_true, minlength=NUM_CLASSES)
for i in range(NUM_CLASSES):
    print(f'  [{i}] {LABEL_NAMES[i]:12s}: {cnt[i]:>9,}  ({cnt[i]/len(y_true)*100:5.2f}%)')


Tap test    : 1,396,358 mau
X_flat shape: (1396358, 37)
X_cnn  shape: (1396358, 37, 1)

Phan bo nhan:
  [0] Benign      :   209,448  (15.00%)
  [1] DDoS        :   598,978  (42.90%)
  [2] DoS         :   193,688  (13.87%)
  [3] BruteForce  :     2,504  ( 0.18%)
  [4] Spoofing    :    87,173  ( 6.24%)
  [5] Recon       :   119,826  ( 8.58%)
  [6] Web-Based   :     4,741  ( 0.34%)
  [7] Mirai       :   180,000  (12.89%)


## Cell 3 - Ham convert (doc state_dict -> Keras -> TFLite)

Doc `state_dict` theo thu tu, gom theo prefix, bo qua `proj.*` va `regressor.*` (chi dung luc train).
Nhan dien kien truc theo ten file. Co them `np_reference` (numpy, quy uoc PyTorch) de kiem chung doc lap.

In [3]:
import numpy as np, torch

def extract_ops(sd):
    """Quet state_dict theo THU TU, gom theo prefix -> list op.
    Bo qua proj.* / regressor.* (train-only, khong thuoc duong inference).
    Tra list: ('conv',W,b) | ('linear',W,b) | ('bn',gamma,beta,mean,var)
    """
    groups = []   # (prefix, dict)
    seen = {}
    for k, v in sd.items():
        kl = k.lower()
        # Bo cac nhanh CHI dung luc train: proj (adaptive tau), projection head (SSL
        # pretrain), regressor (FitNets). Chung khong nam trong duong inference.
        if any(t in kl for t in ('proj', 'regressor', 'hint_net')):
            continue
        if k.endswith('num_batches_tracked'):
            continue
        prefix, last = k.rsplit('.', 1)
        if prefix not in seen:
            seen[prefix] = {}
            groups.append((prefix, seen[prefix]))
        seen[prefix][last] = v
    ops = []
    for prefix, d in groups:
        if 'running_mean' in d:
            ops.append(('bn', d['weight'].numpy(), d['bias'].numpy(),
                        d['running_mean'].numpy(), d['running_var'].numpy(), prefix))
        else:
            W = d['weight'].numpy()
            b = d['bias'].numpy() if 'bias' in d else None
            if W.ndim == 3:   ops.append(('conv', W, b, prefix))
            elif W.ndim == 2: ops.append(('linear', W, b, prefix))
    return ops

def detect_family(name):
    n = name.lower()
    if 'ssl' in n: return 'ssl'
    if 'cnn' in n: return 'cnn'
    if 'dnn' in n: return 'dnn'
    raise ValueError(f'Khong nhan dien duoc kien truc tu ten: {name}')

def unwrap(raw):
    """Xu ly .pth co dict bao ngoai (teacher SSL: encoder_state_dict/classifier_state_dict)."""
    if not isinstance(raw, dict):
        raise ValueError('File .pth khong phai dict')
    # flat state_dict?
    if all(torch.is_tensor(v) for v in raw.values()):
        return raw
    merged = {}
    for key in ['encoder_state_dict', 'classifier_state_dict']:
        if key in raw:
            pre = key.replace('_state_dict', '')
            for k, v in raw[key].items():
                merged[f'{pre}.{k}'] = v
    if merged:
        return merged
    for key in ['model_state', 'state_dict', 'model_state_dict']:
        if key in raw and isinstance(raw[key], dict):
            return raw[key]
    raise ValueError(f'Khong doc duoc state_dict. Keys: {list(raw.keys())[:8]}')

def build_keras(ops, family, num_features=37, num_classes=8):
    """Dung Keras tuong duong + nap trong so tu ops."""
    import tensorflow as tf
    from tensorflow.keras import layers, models

    if family == 'ssl':
        # MLP thuan: moi Linear + ReLU, tru Linear CUOI (khong ReLU)
        lin = [o for o in ops if o[0] == 'linear']
        inp = layers.Input(shape=(num_features,), name='features'); x = inp
        for i, (_, W, b, _) in enumerate(lin):
            last = (i == len(lin) - 1)
            x = layers.Dense(W.shape[0], use_bias=b is not None)(x)
            if not last: x = layers.ReLU()(x)
        m = models.Model(inp, x)
        dense = [l for l in m.layers if isinstance(l, layers.Dense)]
        assert len(dense) == len(lin)
        for (_, W, b, _), l in zip(lin, dense):
            l.set_weights([W.T.copy(), b] if b is not None else [W.T.copy()])
        return m

    if family == 'dnn':
        # [Linear -> ReLU -> BN] xN  +  Linear cuoi
        lin = [o for o in ops if o[0] == 'linear']; bns = [o for o in ops if o[0] == 'bn']
        n_blocks = len(bns)
        assert len(lin) == n_blocks + 1, f'lin={len(lin)} bn={n_blocks}'
        inp = layers.Input(shape=(num_features,), name='features'); x = inp
        for i in range(n_blocks):
            x = layers.Dense(lin[i][1].shape[0])(x)
            x = layers.ReLU()(x)
            x = layers.BatchNormalization(epsilon=1e-3)(x)
        x = layers.Dense(lin[-1][1].shape[0])(x)
        m = models.Model(inp, x)
        dense = [l for l in m.layers if isinstance(l, layers.Dense)]
        bnl   = [l for l in m.layers if isinstance(l, layers.BatchNormalization)]
        for (_, W, b, _), l in zip(lin, dense):
            l.set_weights([W.T.copy(), b])
        for (_, g, be, mu, var, _), l in zip(bns, bnl):
            l.set_weights([g, be, mu, var])
        return m

    if family == 'cnn':
        # [Conv -> ReLU -> MaxPool -> BN] xN + Flatten + Dense(reindex)
        convs = [o for o in ops if o[0] == 'conv']; bns = [o for o in ops if o[0] == 'bn']
        lin = [o for o in ops if o[0] == 'linear']
        assert len(convs) == len(bns) and len(lin) == 1
        inp = layers.Input(shape=(num_features, 1), name='features'); x = inp
        for (_, W, b, _) in convs:
            x = layers.Conv1D(W.shape[0], kernel_size=W.shape[2], padding='same')(x)
            x = layers.ReLU()(x)
            x = layers.MaxPooling1D(2)(x)
            x = layers.BatchNormalization(epsilon=1e-3)(x)
        x = layers.Flatten()(x)
        x = layers.Dense(lin[0][1].shape[0])(x)
        m = models.Model(inp, x)
        convl = [l for l in m.layers if isinstance(l, layers.Conv1D)]
        bnl   = [l for l in m.layers if isinstance(l, layers.BatchNormalization)]
        densel= [l for l in m.layers if isinstance(l, layers.Dense)]
        for (_, W, b, _), l in zip(convs, convl):
            l.set_weights([np.transpose(W, (2, 1, 0)).copy(), b])   # (out,in,k)->(k,in,out)
        for (_, g, be, mu, var, _), l in zip(bns, bnl):
            l.set_weights([g, be, mu, var])
        # REINDEX flatten: torch (B,C,L) idx c*L+l ; keras (B,L,C) idx l*C+c
        _, Wl, bl, _ = lin[0]
        C = convs[-1][1].shape[0]
        L = Wl.shape[1] // C
        perm = np.array([(i % C) * L + (i // C) for i in range(Wl.shape[1])], dtype=np.int64)
        densel[0].set_weights([Wl.T[perm].copy(), bl])
        return m
    raise ValueError(family)

def to_tflite(keras_model, int8=False):
    """int8=True -> dynamic range quantization (trong so int8, kich hoat float)."""
    import tensorflow as tf, contextlib, io
    conv = tf.lite.TFLiteConverter.from_keras_model(keras_model)
    if int8:
        conv.optimizations = [tf.lite.Optimize.DEFAULT]   # dynamic range: weight int8
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):      # chan log "Saved artifact ..."
        blob = conv.convert()
    return blob

def tflite_predict(tflite_bytes, X, batch=512):
    import tensorflow as tf
    it = tf.lite.Interpreter(model_content=tflite_bytes)
    inp = it.get_input_details()[0]; out = it.get_output_details()[0]
    outs = []
    n = len(X)
    for s in range(0, n, batch):
        xb = X[s:s+batch].astype(np.float32)
        it.resize_tensor_input(inp['index'], xb.shape); it.allocate_tensors()
        it.set_tensor(inp['index'], xb); it.invoke()
        outs.append(it.get_tensor(out['index']).copy())
    return np.concatenate(outs, 0)

# ─────────── Reference forward bang NUMPY theo DUNG quy uoc PyTorch ───────────
def np_reference(ops, family, X):
    """X: (N,37) cho dnn/ssl ; (N,1,37) channels-first cho cnn. Tra logits."""
    relu = lambda a: np.maximum(a, 0.0)
    def bn(a, g, b, mu, var, axis):
        sh = [1]*a.ndim; sh[axis] = -1
        return (a - mu.reshape(sh)) / np.sqrt(var.reshape(sh) + 1e-3) * g.reshape(sh) + b.reshape(sh)
    if family == 'ssl':
        lin = [o for o in ops if o[0]=='linear']; h = X
        for i,(_,W,b,_) in enumerate(lin):
            h = h @ W.T + (b if b is not None else 0)
            if i < len(lin)-1: h = relu(h)
        return h
    if family == 'dnn':
        lin=[o for o in ops if o[0]=='linear']; bns=[o for o in ops if o[0]=='bn']; h=X
        for i in range(len(bns)):
            h = relu(h @ lin[i][1].T + lin[i][2])
            h = bn(h, bns[i][1], bns[i][2], bns[i][3], bns[i][4], 1)
        return h @ lin[-1][1].T + lin[-1][2]
    if family == 'cnn':
        convs=[o for o in ops if o[0]=='conv']; bns=[o for o in ops if o[0]=='bn']
        lin=[o for o in ops if o[0]=='linear']; h=X            # (N,C,L)
        for (_,W,b,_),(_,g,be,mu,var,_) in zip(convs,bns):
            k = W.shape[2]; pad = k//2
            hp = np.pad(h, ((0,0),(0,0),(pad,pad)))
            N,Cin,Lp = hp.shape; Cout = W.shape[0]; Lout = Lp-2*pad
            win = np.lib.stride_tricks.sliding_window_view(hp, k, axis=2)   # (N,Cin,Lout,k)
            o = np.einsum('nclk,ock->nol', win, W) + b.reshape(1,-1,1)
            o = relu(o)
            Lp2 = (o.shape[2]//2)*2
            o = o[:,:,:Lp2].reshape(N,Cout,Lp2//2,2).max(-1)                 # MaxPool(2)
            h = bn(o, g, be, mu, var, 1)
        flat = h.reshape(h.shape[0], -1)                                     # PyTorch: c*L+l
        return flat @ lin[0][1].T + lin[0][2]
    raise ValueError(family)

def count_params_from_ops(ops):
    """Dem so tham so (khong tinh proj vi da bi loc, khong tinh buffer running_mean/var)
    -- khop voi count_params() cua PyTorch."""
    n = 0
    for o in ops:
        if o[0] == 'bn':
            n += o[1].size + o[2].size           # gamma + beta
        else:
            n += o[1].size + (o[2].size if o[2] is not None else 0)
    return n

def ops_summary(ops):
    d = {}
    for o in ops:
        d[o[0]] = d.get(o[0], 0) + 1
    return ', '.join(f'{k}={v}' for k, v in d.items())

# ── Chu ky kien truc mong doi: kiem tra DAY DU tung chieu ────────────────────
# Bat moi sai lech (thieu/thua lop, sai chieu, lot nham projection head...).
def expected_signature(family, kind):
    """Tra list op mong doi: ('linear', out, in) / ('conv', out, in, k) / ('bn', ch)."""
    if family == 'ssl':
        if kind == 'Teacher':      # encoder 37-512-512-256-256 + classifier 256-256-8
            dims = [(512,37),(512,512),(256,512),(256,256),(256,256),(8,256)]
        else:                       # encoder 37-128-64-64 + classifier 64-32-8
            dims = [(128,37),(64,128),(64,64),(32,64),(8,32)]
        return [('linear',) + d for d in dims]
    if family == 'dnn':
        h = [256,256,256] if kind == 'Teacher' else [64,64,64]
        sig, d = [], 37
        for u in h:
            sig += [('linear', u, d), ('bn', u)]
            d = u
        sig += [('linear', 8, d)]
        return sig
    if family == 'cnn':
        f = [128,128,256] if kind == 'Teacher' else [32,32,64]
        sig, ch, L = [], 1, 37
        for u in f:
            sig += [('conv', u, ch, 3), ('bn', u)]
            ch, L = u, L // 2
        sig += [('linear', 8, L * ch)]
        return sig
    raise ValueError(family)

def actual_signature(ops):
    sig = []
    for o in ops:
        if o[0] == 'bn':
            sig.append(('bn', int(o[1].shape[0])))
        elif o[0] == 'conv':
            W = o[1]; sig.append(('conv', int(W.shape[0]), int(W.shape[1]), int(W.shape[2])))
        else:
            W = o[1]; sig.append(('linear', int(W.shape[0]), int(W.shape[1])))
    return sig

def check_arch(ops, family, kind, name):
    exp = expected_signature(family, kind)
    act = actual_signature(ops)
    if act != exp:
        msg = [f'{name}: kien truc KHONG khop mong doi.']
        msg.append(f'  Mong doi ({len(exp)} lop): {exp}')
        msg.append(f'  Thuc te  ({len(act)} lop): {act}')
        raise ValueError('\n'.join(msg))

print('Ham convert san sang.')


Ham convert san sang.


## Cell 4 - Convert tat ca sang TFLite FP32 + INT8 dynamic

Moi model: doc `.pth` -> dung Keras -> kiem chung numpy reference -> xuat 2 file `.tflite`.

In [4]:
import tensorflow as tf
tf.get_logger().setLevel('ERROR')

converted = []      # moi phan tu: dict thong tin 1 model

print(f'{"MODEL":34s} {"KT":4s} {"CAU TRUC":26s} {"REF-DIFF":>10s}  TRANG THAI')
print('-' * 92)

for fn, label, kind in PTH_LIST:
    path = os.path.join(PTH_DIR, fn)
    if not os.path.exists(path):
        print(f'{fn[:34]:34s} {"":4s} {"":26s} {"":>10s}  BO QUA (khong tim thay)')
        continue
    try:
        family = detect_family(fn)
        raw = torch.load(path, map_location='cpu', weights_only=False)
        sd  = unwrap(raw)
        ops = extract_ops(sd)
        check_arch(ops, family, kind, fn)

        keras_model = build_keras(ops, family, NUM_FEATURES, NUM_CLASSES)

        # Kiem chung doc lap: numpy reference (quy uoc PyTorch) vs Keras
        n_chk = min(256, len(y_true))
        if family == 'cnn':
            x_ref = X_flat[:n_chk].reshape(-1, 1, NUM_FEATURES).astype(np.float64)
            x_k   = X_cnn[:n_chk]
        else:
            x_ref = X_flat[:n_chk].astype(np.float64)
            x_k   = X_flat[:n_chk]
        ref = np_reference(ops, family, x_ref).astype(np.float32)
        out_k = keras_model.predict(x_k, verbose=0)
        ref_diff = float(np.abs(ref - out_k).max())

        tfl_fp32 = to_tflite(keras_model, int8=False)
        tfl_int8 = to_tflite(keras_model, int8=True)

        base = fn.replace('.pth', '')
        p32 = os.path.join(DIR_MODEL, base + '_fp32.tflite')
        p8  = os.path.join(DIR_MODEL, base + '_int8.tflite')
        with open(p32, 'wb') as f: f.write(tfl_fp32)
        with open(p8,  'wb') as f: f.write(tfl_int8)

        n_params = count_params_from_ops(ops)
        has_proj = any(k.startswith('proj') or '.proj' in k for k in sd.keys())

        converted.append(dict(
            file=fn, label=label, kind=kind, family=family,
            params=n_params,
            size_pth_disk_kb=os.path.getsize(path)/1024,
            size_before_kb=n_params*4/1024,          # PyTorch fp32, da loai proj
            size_fp32_kb=len(tfl_fp32)/1024,
            size_int8_kb=len(tfl_int8)/1024,
            path_fp32=p32, path_int8=p8,
            has_proj=has_proj, ref_diff=ref_diff,
        ))
        status = 'OK' if ref_diff < 1e-3 else f'CANH BAO ref_diff={ref_diff:.2e}'
        print(f'{fn[:34]:34s} {family.upper():4s} {ops_summary(ops):26s} {ref_diff:10.2e}  {status}')
        del keras_model; gc.collect()
    except Exception as e:
        print(f'{fn[:34]:34s} {"":4s} {"":26s} {"":>10s}  LOI: {e}')

print('-' * 92)
print(f'Da convert {len(converted)}/{len(PTH_LIST)} model (moi model 2 file: FP32 + INT8).')
print()
print('Model co lop proj (adaptive tau) - da LOAI BO proj khoi TFLite va khoi phep do size:')
for c in converted:
    if c['has_proj']:
        print(f'  - {c["label"]}  (params sau khi bo proj: {c["params"]:,})')


MODEL                              KT   CAU TRUC                     REF-DIFF  TRANG THAI
--------------------------------------------------------------------------------------------


2026-08-06 02:49:45.610424: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
W0000 00:00:1785984586.727816      58 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1785984586.727885      58 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1785984586.737975      58 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
W0000 00:00:1785984587.731985      58 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1785984587.732035      58 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.


teacher_ssl_cl_dnn_37ft.pth        SSL  linear=6                     7.32e-04  OK


W0000 00:00:1785984589.619526      58 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1785984589.619609      58 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
W0000 00:00:1785984590.622302      58 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1785984590.622389      58 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.


teacher_dnn_37ft.pth               DNN  linear=4, bn=3               3.97e-05  OK


...

student_cnn_adapt_t_classw3_37ft.p CNN  conv=3, bn=3, linear=1       7.63e-06  OK


W0000 00:00:1785984627.773017      58 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1785984627.773066      58 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
W0000 00:00:1785984628.900151      58 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1785984628.900202      58 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.


student_cnn_vanilla_kd_classw3_37f CNN  conv=3, bn=3, linear=1       9.54e-06  OK


W0000 00:00:1785984631.253788      58 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1785984631.253861      58 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
W0000 00:00:1785984632.364052      58 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1785984632.364143      58 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.


student_cnn_fitnets_classw3_37ft.p CNN  conv=3, bn=3, linear=1       9.54e-06  OK
--------------------------------------------------------------------------------------------
Da convert 15/15 model (moi model 2 file: FP32 + INT8).

Model co lop proj (adaptive tau) - da LOAI BO proj khoi TFLite va khoi phep do size:
  - SSL-CL-DNN RAID adaptive tau  (params sau khi bo proj: 19,624)
  - DNN RAID adaptive tau  (params sau khi bo proj: 11,656)
  - CNN RAID adaptive tau  (params sau khi bo proj: 11,752)


## Cell 5 - Danh gia tren tap test (FP32 va INT8)

In [5]:
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             classification_report, confusion_matrix)

def softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def eval_tflite(tflite_path, family):
    with open(tflite_path, 'rb') as f:
        blob = f.read()
    X = X_cnn if family == 'cnn' else X_flat
    t0 = time.perf_counter()
    logits = tflite_predict(blob, X, batch=BATCH_SIZE)
    dt = time.perf_counter() - t0
    probs = softmax(logits.astype(np.float64))
    preds = logits.argmax(1)
    return dict(
        preds=preds, probs=probs,
        acc=accuracy_score(y_true, preds),
        macro_f1=f1_score(y_true, preds, average='macro', zero_division=0),
        f1_per_class=f1_score(y_true, preds, average=None, zero_division=0,
                              labels=list(range(NUM_CLASSES))),
        macro_auc=roc_auc_score(y_true, probs, multi_class='ovr', average='macro'),
        latency_us=dt / len(y_true) * 1e6,
    )

results = {'fp32': {}, 'int8': {}}
print('Dang danh gia ... (moi model chay tren toan bo tap test)')
print()
for c in converted:
    for prec, key in [('fp32', 'path_fp32'), ('int8', 'path_int8')]:
        r = eval_tflite(c[key], c['family'])
        results[prec][c['file']] = r
        print(f'  {c["label"]:34s} {prec.upper():5s}  acc={r["acc"]*100:7.4f}%  macroF1={r["macro_f1"]*100:7.4f}%')
    gc.collect()
print()
print('Danh gia xong.')


Dang danh gia ... (moi model chay tren toan bo tap test)



INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


  Teacher SSL-CL-DNN                 FP32   acc=85.5793%  macroF1=70.9256%
  Teacher SSL-CL-DNN                 INT8   acc=83.5688%  macroF1=67.6062%
  Teacher DNN                        FP32   acc=85.7090%  macroF1=71.4273%
  Teacher DNN                        INT8   acc=83.9599%  macroF1=67.3405%
  Teacher CNN                        FP32   acc=84.6653%  macroF1=67.8706%
  Teacher CNN                        INT8   acc=84.6720%  macroF1=67.9098%
  SSL-CL-DNN RAID fixed tau          FP32   acc=82.2082%  macroF1=69.3857%
  SSL-CL-DNN RAID fixed tau          INT8   acc=78.7917%  macroF1=65.8268%
  SSL-CL-DNN RAID adaptive tau       FP32   acc=82.6174%  macroF1=69.9203%
  SSL-CL-DNN RAID adaptive tau       INT8   acc=81.8193%  macroF1=67.6289%
  SSL-CL-DNN VanillaKD               FP32   acc=84.6408%  macroF1=70.1893%
  SSL-CL-DNN VanillaKD               INT8   acc=81.3766%  macroF1=64.5463%
  SSL-CL-DNN FitNets                 FP32   acc=84.3807%  macroF1=70.0540%
  SSL-CL-DNN FitNets     

## Cell 6 - Ket qua FP32 (in doc)

In [6]:
def block(c, prec):
    r = results[prec][c['file']]
    size_after = c['size_fp32_kb'] if prec == 'fp32' else c['size_int8_kb']
    ratio = c['size_before_kb'] / size_after if size_after > 0 else float('nan')
    saved = (1 - size_after / c['size_before_kb']) * 100
    lines = []
    lines.append('=' * 60)
    lines.append(f'{prec.upper()}  |  {c["label"]}')
    lines.append('=' * 60)
    lines.append(f'  File goc            : {c["file"]}')
    lines.append(f'  Kien truc           : {c["family"].upper()}')
    lines.append(f'  Loai                : {c["kind"]}')
    lines.append(f'  Accuracy            : {r["acc"]*100:.4f} %')
    lines.append(f'  Macro F1            : {r["macro_f1"]*100:.4f} %')
    lines.append(f'  Macro AUC           : {r["macro_auc"]:.6f}')
    lines.append(f'  So tham so          : {c["params"]:,}')
    if c['has_proj']:
        lines.append(f'  Ghi chu             : da loai bo lop proj (adaptive tau) khoi model va phep do size')
    lines.append(f'  Size truoc nen      : {c["size_before_kb"]:.2f} KB   (PyTorch FP32)')
    lines.append(f'  Size sau nen        : {size_after:.2f} KB   (TFLite {prec.upper()})')
    lines.append(f'  Ty le nen           : {ratio:.2f}x   (giam {saved:.2f} %)')
    lines.append(f'  Thoi gian suy luan  : {r["latency_us"]:.2f} us / mau')
    lines.append('')
    return '\n'.join(lines)

txt_fp32 = []
print()
print('#' * 60)
print('#  KET QUA TFLITE FP32')
print('#' * 60)
print()
for c in converted:
    s = block(c, 'fp32')
    txt_fp32.append(s)
    print(s)



############################################################
#  KET QUA TFLITE FP32
############################################################

FP32  |  Teacher SSL-CL-DNN
  File goc            : teacher_ssl_cl_dnn_37ft.pth
  Kien truc           : SSL
  Loai                : Teacher
  Accuracy            : 85.5793 %
  Macro F1            : 70.9256 %
  Macro AUC           : 0.979741
  So tham so          : 547,080
  Size truoc nen      : 2137.03 KB   (PyTorch FP32)
  Size sau nen        : 2139.96 KB   (TFLite FP32)
  Ty le nen           : 1.00x   (giam -0.14 %)
  Thoi gian suy luan  : 19.26 us / mau

FP32  |  Teacher DNN
  File goc            : teacher_dnn_37ft.pth
  Kien truc           : DNN
  Loai                : Teacher
  Accuracy            : 85.7090 %
  Macro F1            : 71.4273 %
  Macro AUC           : 0.980434
  So tham so          : 144,904
  Size truoc nen      : 566.03 KB   (PyTorch FP32)
  Size sau nen        : 562.47 KB   (TFLite FP32)
  Ty le nen           : 1.01x 

## Cell 7 - Ket qua INT8 dynamic (in doc)

In [7]:
txt_int8 = []
print()
print('#' * 60)
print('#  KET QUA TFLITE INT8 (dynamic range)')
print('#' * 60)
print()
for c in converted:
    s = block(c, 'int8')
    txt_int8.append(s)
    print(s)



############################################################
#  KET QUA TFLITE INT8 (dynamic range)
############################################################

INT8  |  Teacher SSL-CL-DNN
  File goc            : teacher_ssl_cl_dnn_37ft.pth
  Kien truc           : SSL
  Loai                : Teacher
  Accuracy            : 83.5688 %
  Macro F1            : 67.6062 %
  Macro AUC           : 0.974347
  So tham so          : 547,080
  Size truoc nen      : 2137.03 KB   (PyTorch FP32)
  Size sau nen        : 563.71 KB   (TFLite INT8)
  Ty le nen           : 3.79x   (giam 73.62 %)
  Thoi gian suy luan  : 23.09 us / mau

INT8  |  Teacher DNN
  File goc            : teacher_dnn_37ft.pth
  Kien truc           : DNN
  Loai                : Teacher
  Accuracy            : 83.9599 %
  Macro F1            : 67.3405 %
  Macro AUC           : 0.971551
  So tham so          : 144,904
  Size truoc nen      : 566.03 KB   (PyTorch FP32)
  Size sau nen        : 153.91 KB   (TFLite INT8)
  Ty le nen    

## Cell 8 - Bang tong hop (doc, FP32 truoc - INT8 sau)

In [8]:
def summary_lines(prec):
    out = []
    out.append('=' * 60)
    out.append(f'TONG HOP - TFLite {prec.upper()}')
    out.append('=' * 60)
    for c in converted:
        r = results[prec][c['file']]
        size_after = c['size_fp32_kb'] if prec == 'fp32' else c['size_int8_kb']
        out.append('')
        out.append(f'{c["label"]}')
        out.append(f'    Accuracy       : {r["acc"]*100:.4f} %')
        out.append(f'    Macro F1       : {r["macro_f1"]*100:.4f} %')
        out.append(f'    Macro AUC      : {r["macro_auc"]:.6f}')
        out.append(f'    Size truoc     : {c["size_before_kb"]:.2f} KB')
        out.append(f'    Size sau       : {size_after:.2f} KB')
        out.append(f'    Ty le nen      : {c["size_before_kb"]/size_after:.2f}x')
    out.append('')
    return out

lines = []
lines += summary_lines('fp32')
lines += summary_lines('int8')
text = '\n'.join(lines)
print(text)

with open(os.path.join(OUT_DIR, 'summary_fp32_int8.txt'), 'w') as f:
    f.write(text)

# Bang CSV de tien loc
rows = []
for prec in ['fp32', 'int8']:
    for c in converted:
        r = results[prec][c['file']]
        size_after = c['size_fp32_kb'] if prec == 'fp32' else c['size_int8_kb']
        rows.append(dict(precision=prec, model=c['label'], file=c['file'],
                         family=c['family'], kind=c['kind'],
                         accuracy=r['acc'], macro_f1=r['macro_f1'], macro_auc=r['macro_auc'],
                         params=c['params'],
                         size_before_kb=round(c['size_before_kb'], 2),
                         size_after_kb=round(size_after, 2),
                         compression_ratio=round(c['size_before_kb']/size_after, 3),
                         latency_us=round(r['latency_us'], 3)))
pd.DataFrame(rows).to_csv(os.path.join(OUT_DIR, 'summary_all.csv'), index=False)
print()
print('Da luu: summary_fp32_int8.txt, summary_all.csv')


TONG HOP - TFLite FP32

Teacher SSL-CL-DNN
    Accuracy       : 85.5793 %
    Macro F1       : 70.9256 %
    Macro AUC      : 0.979741
    Size truoc     : 2137.03 KB
    Size sau       : 2139.96 KB
    Ty le nen      : 1.00x

Teacher DNN
    Accuracy       : 85.7090 %
    Macro F1       : 71.4273 %
    Macro AUC      : 0.980434
    Size truoc     : 566.03 KB
    Size sau       : 562.47 KB
    Ty le nen      : 1.01x

Teacher CNN
    Accuracy       : 84.6653 %
    Macro F1       : 67.8706 %
    Macro AUC      : 0.976573
    Size truoc     : 615.53 KB
    Size sau       : 624.91 KB
    Ty le nen      : 0.98x

SSL-CL-DNN RAID fixed tau
    Accuracy       : 82.2082 %
    Macro F1       : 69.3857 %
    Macro AUC      : 0.977471
    Size truoc     : 76.66 KB
    Size sau       : 79.28 KB
    Ty le nen      : 0.97x

SSL-CL-DNN RAID adaptive tau
    Accuracy       : 82.6174 %
    Macro F1       : 69.9203 %
    Macro AUC      : 0.978008
    Size truoc     : 76.66 KB
    Size sau       : 79.28 K

## Cell 9 - Luu classification report + ma tran nham lan + bieu do

- `reports/`   : classification report + size truoc/sau + AUC
- `confusion/` : ma tran nham lan (so mau va %)
- `charts/`    : bieu do cot accuracy (recall) va F1 theo tung nhan

In [9]:
def save_report(c, prec):
    r = results[prec][c['file']]
    size_after = c['size_fp32_kb'] if prec == 'fp32' else c['size_int8_kb']
    base = c['file'].replace('.pth', '') + '_' + prec

    # ── 1. Classification report ─────────────────────────────────────────────
    rep = classification_report(y_true, r['preds'], labels=list(range(NUM_CLASSES)),
                                target_names=CLASS_NAMES, digits=4, zero_division=0)
    L = []
    L.append('=' * 66)
    L.append(f'{c["label"]}   |   TFLite {prec.upper()}')
    L.append('=' * 66)
    L.append(f'File goc          : {c["file"]}')
    L.append(f'Kien truc         : {c["family"].upper()}')
    L.append(f'Loai              : {c["kind"]}')
    L.append('')
    L.append('CLASSIFICATION REPORT')
    L.append('-' * 66)
    L.append(rep)
    L.append('-' * 66)
    L.append(f'Accuracy          : {r["acc"]*100:.4f} %')
    L.append(f'Macro F1          : {r["macro_f1"]*100:.4f} %')
    L.append(f'Macro AUC (OvR)   : {r["macro_auc"]:.6f}')
    L.append('')
    L.append('KICH THUOC MO HINH')
    L.append('-' * 66)
    L.append(f'So tham so        : {c["params"]:,}')
    if c['has_proj']:
        L.append('Ghi chu           : da loai bo lop proj (adaptive tau) khoi model va phep do size')
    L.append(f'Size truoc nen    : {c["size_before_kb"]:.2f} KB  (PyTorch FP32)')
    L.append(f'Size sau nen      : {size_after:.2f} KB  (TFLite {prec.upper()})')
    L.append(f'Ty le nen         : {c["size_before_kb"]/size_after:.2f}x')
    L.append(f'Giam              : {(1-size_after/c["size_before_kb"])*100:.2f} %')
    L.append(f'File .pth tren dia: {c["size_pth_disk_kb"]:.2f} KB')
    L.append(f'Suy luan          : {r["latency_us"]:.2f} us / mau')
    L.append('')
    L.append('AUC TUNG NHAN (One-vs-Rest)')
    L.append('-' * 66)
    for i in range(NUM_CLASSES):
        yb = (y_true == i).astype(int)
        try:
            auc_i = roc_auc_score(yb, r['probs'][:, i])
            L.append(f'  [{i}] {LABEL_NAMES[i]:12s}: {auc_i:.6f}')
        except Exception:
            L.append(f'  [{i}] {LABEL_NAMES[i]:12s}: khong tinh duoc')
    with open(os.path.join(DIR_REPORT, f'report_{base}.txt'), 'w') as f:
        f.write('\n'.join(L))

    # ── 2. Ma tran nham lan: so mau + % ──────────────────────────────────────
    cm = confusion_matrix(y_true, r['preds'], labels=list(range(NUM_CLASSES)))
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
    CL = []
    CL.append(f'MA TRAN NHAM LAN - {c["label"]} - TFLite {prec.upper()}')
    CL.append('')
    CL.append('A. SO MAU  (hang = nhan that, cot = du doan)')
    CL.append('-' * 96)
    CL.append(f'{"":14s}' + ''.join(f'{n[:10]:>11s}' for n in CLASS_NAMES) + f'{"Tong":>11s}')
    for i in range(NUM_CLASSES):
        CL.append(f'{CLASS_NAMES[i]:14s}' + ''.join(f'{cm[i,j]:>11,}' for j in range(NUM_CLASSES))
                  + f'{cm[i].sum():>11,}')
    CL.append('')
    CL.append('B. PHAN TRAM theo hang (%)')
    CL.append('-' * 96)
    CL.append(f'{"":14s}' + ''.join(f'{n[:10]:>11s}' for n in CLASS_NAMES))
    for i in range(NUM_CLASSES):
        CL.append(f'{CLASS_NAMES[i]:14s}' + ''.join(f'{cm_pct[i,j]:>10.2f}%' for j in range(NUM_CLASSES)))
    CL.append('')
    CL.append('C. CHI TIET TUNG O  (so mau / % theo hang)')
    CL.append('-' * 96)
    for i in range(NUM_CLASSES):
        CL.append(f'  Nhan that: {CLASS_NAMES[i]}  (tong {cm[i].sum():,} mau)')
        for j in range(NUM_CLASSES):
            if cm[i, j] > 0:
                mark = '  <-- dung' if i == j else ''
                CL.append(f'      -> du doan {CLASS_NAMES[j]:12s}: {cm[i,j]:>9,}  ({cm_pct[i,j]:6.2f} %){mark}')
    with open(os.path.join(DIR_CM, f'cm_{base}.txt'), 'w') as f:
        f.write('\n'.join(CL))
    pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(
        os.path.join(DIR_CM, f'cm_{base}_counts.csv'))
    pd.DataFrame(np.round(cm_pct, 4), index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(
        os.path.join(DIR_CM, f'cm_{base}_percent.csv'))

    # ── 3. Bieu do cot: accuracy (recall) va F1 theo tung nhan ───────────────
    recall = np.diag(cm) / np.maximum(cm.sum(axis=1), 1) * 100
    f1c    = r['f1_per_class'] * 100
    fig, ax = plt.subplots(figsize=(12, 6))
    xpos = np.arange(NUM_CLASSES); w = 0.38
    b1 = ax.bar(xpos - w/2, recall, w, label='Accuracy theo nhan (recall)',
                color='#4C9BE8', edgecolor='white')
    b2 = ax.bar(xpos + w/2, f1c, w, label='F1 theo nhan',
                color='#E84C4C', edgecolor='white')
    for bars in (b1, b2):
        for bar in bars:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.2,
                    f'{bar.get_height():.1f}', ha='center', fontsize=8)
    ax.set_xticks(xpos); ax.set_xticklabels(CLASS_NAMES, rotation=30, ha='right')
    ax.set_ylim(0, 115); ax.set_ylabel('%')
    ax.set_title(f'{c["label"]} - TFLite {prec.upper()}\n'
                 f'Accuracy = {r["acc"]*100:.2f} %   |   Macro F1 = {r["macro_f1"]*100:.2f} %   |   '
                 f'Size {c["size_before_kb"]:.1f} KB -> {size_after:.1f} KB')
    ax.legend(loc='lower right', fontsize=9); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(DIR_CHART, f'chart_{base}.png'), dpi=140, bbox_inches='tight')
    plt.close(fig)

print('Dang luu report / ma tran nham lan / bieu do ...')
for prec in ['fp32', 'int8']:
    for c in converted:
        save_report(c, prec)
    print(f'  Xong {prec.upper()}')

# Ghi them 2 file tong hop chi tiet da in o Cell 6 va 7
with open(os.path.join(OUT_DIR, 'ketqua_fp32.txt'), 'w') as f:
    f.write('\n'.join(txt_fp32))
with open(os.path.join(OUT_DIR, 'ketqua_int8.txt'), 'w') as f:
    f.write('\n'.join(txt_int8))

print()
print('Da luu xong toan bo ket qua.')


Dang luu report / ma tran nham lan / bieu do ...
  Xong FP32
  Xong INT8

Da luu xong toan bo ket qua.


## Cell 10 - Bieu do tong hop so sanh cac model

In [10]:
labels = [c['label'] for c in converted]
yp = np.arange(len(converted))

fig, axes = plt.subplots(1, 3, figsize=(19, max(5, 0.42*len(converted)+2)))

# 1. Accuracy
a32 = [results['fp32'][c['file']]['acc']*100 for c in converted]
a8  = [results['int8'][c['file']]['acc']*100 for c in converted]
axes[0].barh(yp-0.2, a32, 0.4, label='FP32', color='#4C9BE8')
axes[0].barh(yp+0.2, a8,  0.4, label='INT8', color='#E84C4C')
axes[0].set_yticks(yp); axes[0].set_yticklabels(labels, fontsize=8)
axes[0].set_xlabel('%'); axes[0].set_title('Accuracy'); axes[0].legend(fontsize=8)
axes[0].grid(axis='x', alpha=0.3); axes[0].invert_yaxis()

# 2. Macro F1
f32 = [results['fp32'][c['file']]['macro_f1']*100 for c in converted]
f8  = [results['int8'][c['file']]['macro_f1']*100 for c in converted]
axes[1].barh(yp-0.2, f32, 0.4, label='FP32', color='#4C9BE8')
axes[1].barh(yp+0.2, f8,  0.4, label='INT8', color='#E84C4C')
axes[1].set_yticks(yp); axes[1].set_yticklabels([], fontsize=8)
axes[1].set_xlabel('%'); axes[1].set_title('Macro F1'); axes[1].legend(fontsize=8)
axes[1].grid(axis='x', alpha=0.3); axes[1].invert_yaxis()

# 3. Kich thuoc
sb  = [c['size_before_kb'] for c in converted]
s32 = [c['size_fp32_kb'] for c in converted]
s8  = [c['size_int8_kb'] for c in converted]
axes[2].barh(yp-0.26, sb,  0.26, label='Truoc (PyTorch FP32)', color='#9B59B6')
axes[2].barh(yp,      s32, 0.26, label='TFLite FP32',          color='#4C9BE8')
axes[2].barh(yp+0.26, s8,  0.26, label='TFLite INT8',          color='#E84C4C')
axes[2].set_yticks(yp); axes[2].set_yticklabels([], fontsize=8)
axes[2].set_xlabel('KB'); axes[2].set_title('Kich thuoc mo hinh'); axes[2].legend(fontsize=8)
axes[2].set_xscale('log'); axes[2].grid(axis='x', alpha=0.3); axes[2].invert_yaxis()

plt.suptitle('So sanh toan bo mo hinh sau khi convert sang TFLite',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout(rect=[0,0,1,0.99])
p = os.path.join(DIR_CHART, 'tong_hop_tat_ca_model.png')
plt.savefig(p, dpi=140, bbox_inches='tight'); plt.show()
print('Bieu do tong hop ->', p)


Bieu do tong hop -> /kaggle/working/tflite_out/charts/tong_hop_tat_ca_model.png


## Cell 11 - Liet ke file da xuat

In [11]:
total = 0
for root, _, files in os.walk(OUT_DIR):
    rel = os.path.relpath(root, OUT_DIR)
    files = sorted(files)
    if not files:
        continue
    print()
    print(f'[{rel if rel != "." else OUT_DIR}]  ({len(files)} file)')
    for fname in files:
        fp = os.path.join(root, fname)
        sz = os.path.getsize(fp) / 1024
        total += sz
        print(f'   {sz:9.1f} KB   {fname}')
print()
print(f'Tong dung luong xuat ra: {total/1024:.2f} MB')
print(f'Thu muc: {OUT_DIR}')



[/kaggle/working/tflite_out]  (4 file)
         8.9 KB   ketqua_fp32.txt
         8.9 KB   ketqua_int8.txt
         4.8 KB   summary_all.csv
         6.1 KB   summary_fp32_int8.txt

[confusion]  (90 file)
         5.7 KB   cm_student_cnn_adapt_t_classw3_37ft_fp32.txt
         0.4 KB   cm_student_cnn_adapt_t_classw3_37ft_fp32_counts.csv
         0.5 KB   cm_student_cnn_adapt_t_classw3_37ft_fp32_percent.csv
         5.7 KB   cm_student_cnn_adapt_t_classw3_37ft_int8.txt
         0.3 KB   cm_student_cnn_adapt_t_classw3_37ft_int8_counts.csv
         0.5 KB   cm_student_cnn_adapt_t_classw3_37ft_int8_percent.csv
         5.5 KB   cm_student_cnn_fitnets_classw3_37ft_fp32.txt
         0.3 KB   cm_student_cnn_fitnets_classw3_37ft_fp32_counts.csv
         0.5 KB   cm_student_cnn_fitnets_classw3_37ft_fp32_percent.csv
         5.5 KB   cm_student_cnn_fitnets_classw3_37ft_int8.txt
         0.3 KB   cm_student_cnn_fitnets_classw3_37ft_int8_counts.csv
         0.5 KB   cm_student_cnn_fitnets_classw3_